# SHAP — Random Forest Feature Attribution

**Why SHAP instead of Gini importance:**  
The RF feature importance plot in notebook 02 showed which features are used most across all splits (Gini impurity). SHAP goes further: it shows *how* each feature pushes a specific prediction toward cancer or healthy, and *in which direction*.

**TreeExplainer — why it's exact:**  
SHAP for tree models is computed exactly using Shapley values from game theory. For each prediction, it decomposes the output into additive contributions from each feature. No approximation, no sampling — the exact contribution of every feature to every prediction.

**What we produce:**
1. **Summary plot (beeswarm)** — all 38 test samples × 18 features. Each dot is one sample. X-axis = SHAP value (positive = pushes toward cancer). Color = feature value (red = high, blue = low). Answers: which features matter most, and in which direction?
2. **Waterfall plot** — a single cancer sample. Shows the step-by-step breakdown of how each feature moved the prediction from the baseline to the final score. Answers: why did the model call this specific sample cancer?

**Biology check:**  
If SHAP is working correctly, the features that push predictions toward cancer should be:
- High `short_long_ratio` (cancer has more short fragments)
- Low values in the 150–200 bp bins (depleted nucleosomal fragments)
- These should be the top contributors, not arbitrary bin features

## Step 1 — Load RF model + test data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap, pickle, pathlib

ROOT      = pathlib.Path().resolve().parent
SPLIT_DIR = ROOT / 'data' / 'processed' / 'split'

X_train   = np.load(SPLIT_DIR / 'X_train.npy')
X_test    = np.load(SPLIT_DIR / 'X_test.npy')
y_train   = np.load(SPLIT_DIR / 'y_train.npy')
y_test    = np.load(SPLIT_DIR / 'y_test.npy')
feat_cols = np.load(SPLIT_DIR / 'feat_cols.npy', allow_pickle=True).tolist()

with open(ROOT / 'models' / 'rf_final.pkl', 'rb') as f:
    rf_model = pickle.load(f)

print(f'SHAP version : {shap.__version__}')
print(f'X_test       : {X_test.shape}')
print(f'Features     : {feat_cols}')

## Step 2 — Compute SHAP values

In [ ]:
# TreeExplainer uses the training set as background — exact Shapley values
explainer   = shap.TreeExplainer(rf_model, data=X_train, feature_names=feat_cols)
shap_values = explainer(X_test)

# shap_values has shape (38, 18, 2) for binary classification — index 1 = cancer class
print(f'SHAP values shape : {shap_values.shape}')
print(f'Base value (cancer class) : {shap_values[..., 1].base_values[0]:.4f}')
print('Ready to plot.')

## Step 3 — Summary plot (beeswarm)

**How to read it:**  
- Features ranked top-to-bottom by mean |SHAP value| — most important at top
- Each dot = one test sample
- X-axis: SHAP value for cancer class. Positive = pushed toward cancer, negative = pushed toward healthy
- Colour: red = high feature value, blue = low feature value

**What to look for:**  
- `short_long_ratio`: red dots (high SFR) should be on the right (→ cancer). Matches biology.
- Long-fragment bins (150–200 bp): blue dots (low bin density) should be on the right. A depleted nucleosomal peak = fewer long fragments = cancer signal.

In [ ]:
shap_cancer = shap_values[..., 1]   # cancer class SHAP values

plt.figure(figsize=(9, 6))
shap.plots.beeswarm(shap_cancer, max_display=18, show=False)
plt.title('SHAP Summary — Random Forest (cancer class)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'shap_beeswarm_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/shap_beeswarm_rf.png')

## Step 4 — Waterfall plot (single cancer sample)

**How to read it:**  
- Starts at the base value (E[f(x)] = average model output across training set)
- Each feature either adds to (red, →cancer) or subtracts from (blue, →healthy) the prediction
- Ends at f(x) = the model's actual output for this sample

We pick the cancer sample the RF was most confident about — highest predicted cancer probability.

In [ ]:
# Most confident cancer prediction in test set
cancer_idx   = np.where(y_test == 1)[0]
prob_cancer  = rf_model.predict_proba(X_test[cancer_idx])[:, 1]
best_sample  = cancer_idx[np.argmax(prob_cancer)]

print(f'Most confident cancer sample: test index {best_sample}')
print(f'  Predicted cancer probability : {prob_cancer.max():.4f}')
print(f'  True label                   : {"cancer" if y_test[best_sample] == 1 else "healthy"}')

plt.figure(figsize=(9, 6))
shap.plots.waterfall(shap_cancer[best_sample], max_display=18, show=False)
plt.title(f'SHAP Waterfall — Most Confident Cancer Prediction  (p={prob_cancer.max():.3f})',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'shap_waterfall_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/shap_waterfall_rf.png')